In [8]:
import shap
import numpy as np

# Ensure correct feature order
X = X[model.feature_names_in_]

# background sample for SHAP stability
background = X.sample(min(100, len(X)), random_state=42)

# Use LinearExplainer for linear models (Logistic / Linear etc.)
explainer = shap.LinearExplainer(model, background)

shap_values = explainer.shap_values(X)

In [9]:
import matplotlib.pyplot as plt
import os

os.makedirs("../reports/shap", exist_ok=True)

plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.title("SHAP Global Feature Importance")

plt.savefig("../reports/shap/shap_summary.png", bbox_inches="tight")
plt.close()

In [10]:
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.title("Top Features Driving Churn Prediction")

plt.savefig("../reports/shap/top_features.png", bbox_inches="tight")
plt.close()

In [11]:
churned_sample = X[y == 1].iloc[[0]]

plt.figure()
shap.force_plot(
    explainer.expected_value,
    explainer.shap_values(churned_sample),
    churned_sample,
    matplotlib=True,
    show=False
)

plt.savefig("../reports/shap/local_churned.png", bbox_inches="tight")
plt.close()

<Figure size 640x480 with 0 Axes>

In [12]:
retained_sample = X[y == 0].iloc[[0]]

plt.figure()
shap.force_plot(
    explainer.expected_value,
    explainer.shap_values(retained_sample),
    retained_sample,
    matplotlib=True,
    show=False
)

plt.savefig("../reports/shap/local_retained.png", bbox_inches="tight")
plt.close()

In [13]:
from sklearn.inspection import PartialDependenceDisplay

top_features = ["tenure", "monthlycharges"]

fig, ax = plt.subplots(figsize=(10, 5))

PartialDependenceDisplay.from_estimator(
    model,
    X,
    features=top_features,
    ax=ax
)

plt.savefig("../reports/shap/pdp_top_features.png", bbox_inches="tight")
plt.close()

Business Questions & Insights

1. Top 5 churn drivers
Contract type (month-to-month strongest)
Tenure (low tenure → high churn)
Monthly charges (high cost sensitivity)
Tech support absence
Fiber optic service usage

2. High risk segments
New customers (< 6 months tenure)
Month-to-month contracts
High monthly charges users
Fiber optic users without support

3. Pricing strategy (from regression insight)
Reduce entry-level pricing for new users
Bundle services (internet + support discounts)
Incentivize long-term contracts

4. Target 100 customers strategy

Prioritize:

High SHAP churn probability
High revenue customers
Low tenure but high charges

Risk score:

risk_score = churn_probability * monthlycharges

Select top 100 customers based on this score.

5. ROI Calculation

Assumptions:

Retention cost = $50
Churn loss = $500
If model is used:
70 churners correctly identified
Prevented loss = 70 × 500 = $35,000
Cost = 100 × 50 = $5,000
Net gain = $30,000
If random selection:
~20 churners captured
Gain = $10,000
Cost = $5,000
Net = $5,000

Model provides ~6x higher ROI compared to random targeting